# 010 — Exploratory Data Analysis

Inventory the dataset, validate artwork-grouped splits, visualise sample pairs and pixel distributions.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path
project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.config import settings
from scripts.dataset import (
    extract_artwork_id,
    grouped_train_val_test_split,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.visualization import plot_sample_pairs

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Dataset inventory

In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
print(f"Total pairs: {len(pairs)}")
print(f"\nFirst 5 pairs:")
for rgb, ir in pairs[:5]:
    print(f"  RGB: {rgb.name}  |  IR: {ir.name}")

In [ ]:
artwork_ids = [extract_artwork_id(p[0].stem) for p in pairs]
counts = Counter(artwork_ids)

print(f"Unique artworks: {len(counts)}\n")
print(f"{'Artwork':<20} {'Sections':>8}")
print("-" * 30)
for artwork, n in sorted(counts.items()):
    print(f"{artwork:<20} {n:>8}")
print("-" * 30)
print(f"{'Total':<20} {sum(counts.values()):>8}")

## 2. Train / val / test split by artwork

In [ ]:
# Artworks split logic: every artwork ID is a group, and each group is kept
# entirely within a single fold (train, val, or test) to prevent leakage
# between sections of the same painting. Applied uniformly to all artworks,
# including the paint-on-support mockup groups (tblu, tbianco, tbruno,
# tgiallo, trosso, tverde) — so an entire mockup group can end up held out
# in test, even though those groups exist specifically to aid training.
train_pairs, val_pairs, test_pairs = grouped_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    seed=settings.SEED,
)

train_artworks = sorted({extract_artwork_id(p[0].stem) for p in train_pairs})
val_artworks   = sorted({extract_artwork_id(p[0].stem) for p in val_pairs})
test_artworks  = sorted({extract_artwork_id(p[0].stem) for p in test_pairs})

print(f"Train: {len(train_pairs):>4} patches | {len(train_artworks):>2} artworks: {train_artworks}")
print(f"Val:   {len(val_pairs):>4} patches | {len(val_artworks):>2} artworks: {val_artworks}")
print(f"Test:  {len(test_pairs):>4} patches | {len(test_artworks):>2} artworks: {test_artworks}")

In [ ]:
# Artwork and mockups split logic (alternative to the block above): real
# artworks are still grouped and leakage-free as above, but the mockup
# groups listed in settings.MOCKUP_ARTWORK_IDS are split at the individual
# pair level instead of by whole group — only a small fraction
# (mockup_test_ratio, default 5%) goes to test, the rest is spread across
# train/val. This keeps mockup sections available for training while still
# holding out a small sample of each for evaluation. Not used further in
# this notebook — rename to train_pairs/val_pairs/test_pairs above to switch
# the rest of the notebook over to this split.
train_pairs_mockups, val_pairs_mockups, test_pairs_mockups = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    seed=settings.SEED,
)

train_artworks_mockups = sorted({extract_artwork_id(p[0].stem) for p in train_pairs_mockups})
val_artworks_mockups   = sorted({extract_artwork_id(p[0].stem) for p in val_pairs_mockups})
test_artworks_mockups  = sorted({extract_artwork_id(p[0].stem) for p in test_pairs_mockups})

print(f"Train: {len(train_pairs_mockups):>4} patches | {len(train_artworks_mockups):>2} artworks: {train_artworks_mockups}")
print(f"Val:   {len(val_pairs_mockups):>4} patches | {len(val_artworks_mockups):>2} artworks: {val_artworks_mockups}")
print(f"Test:  {len(test_pairs_mockups):>4} patches | {len(test_artworks_mockups):>2} artworks: {test_artworks_mockups}")

In [ ]:
# Verify no artwork appears in more than one fold
assert not (set(train_artworks) & set(val_artworks)),  "Leakage: train ∩ val"
assert not (set(train_artworks) & set(test_artworks)), "Leakage: train ∩ test"
assert not (set(val_artworks)   & set(test_artworks)), "Leakage: val ∩ test"
assert len(train_pairs) + len(val_pairs) + len(test_pairs) == len(pairs), "Total mismatch"
print("All split integrity checks passed.")

## 3. Sample RGB / IR pair visualisation

In [ ]:
random.seed(settings.SEED)
sample = random.sample(pairs, min(5, len(pairs)))

rgb_imgs = [np.array(Image.open(p[0]).convert("RGB")) / 255.0 for p in sample]
ir_imgs  = [np.array(Image.open(p[1]).convert("L"))  / 255.0 for p in sample]

fig = plot_sample_pairs(rgb_imgs, ir_imgs, n=5)
plt.show()

## 4. Pixel intensity distributions

In [ ]:
rgb_arr = np.stack(rgb_imgs)   # (N, H, W, 3)
ir_arr  = np.stack(ir_imgs)    # (N, H, W)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Pixel intensity histograms (sample of 5 pairs)")

for i, (ch, col) in enumerate(zip(["R", "G", "B"], ["red", "green", "blue"])):
    axes[i].hist(rgb_arr[..., i].ravel(), bins=64, color=col, alpha=0.75)
    axes[i].set_title(f"Channel {ch}")
    axes[i].set_xlabel("Intensity")

axes[3].hist(ir_arr.ravel(), bins=64, color="gray", alpha=0.75)
axes[3].set_title("IR")
axes[3].set_xlabel("Intensity")

plt.tight_layout()
plt.show()

## 5. Image dimensions check

In [ ]:
# RGB/IR size agreement is now enforced for every pair by
# load_image_pairs() itself (it raises ValueError on any mismatch), so
# `pairs` above is already guaranteed consistent. This cell is purely
# descriptive: it reports the distribution of image sizes present.
sizes = {}
for rgb_path, _ in pairs:
    rgb_size = Image.open(rgb_path).size  # (W, H)
    sizes[rgb_size] = sizes.get(rgb_size, 0) + 1

print("Unique image sizes (W×H) across all pairs:")
for size, count in sorted(sizes.items()):
    print(f"  {size[0]}×{size[1]}: {count} images")
print(f"\nPATCH_MULTIPLE = {settings.PATCH_MULTIPLE}")
print(f"400 mod {settings.PATCH_MULTIPLE} = {400 % settings.PATCH_MULTIPLE} (0 = no padding needed at training time)")